# Handwritten Digit Recognition with MNIST

This guided notebook walks through data collection, preprocessing, model building, training, evaluation, and prediction. Run the cells from top to bottom. The first data-loading cell downloads MNIST if it is not already cached.

## 1. Imports and data collection

MNIST contains 60,000 training images and 10,000 test images. Every image is a 28 by 28 grayscale picture of a digit from 0 to 9.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from src import tensorflow_config  # Configure quieter TensorFlow logs first.
from tensorflow.keras.utils import to_categorical
from src.data_loader import load_mnist_data
from src.preprocess import preprocess_data
from src.model import build_model
from src.predict import predict_image

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = load_mnist_data()
print('Training:', x_train_raw.shape, y_train_raw.shape)
print('Test:', x_test_raw.shape, y_test_raw.shape)

## 2. Explore the images

Plot a few examples to see the pixel format and the label associated with each picture.

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(10, 5))
for index, axis in enumerate(axes.flat):
    axis.imshow(x_train_raw[index], cmap='gray')
    axis.set_title(f'Label: {y_train_raw[index]}')
    axis.axis('off')
plt.tight_layout()
plt.show()

## 3. Preprocess the data

Pixel values are divided by 255 so they fall between 0 and 1. We add a single channel dimension for compatibility with both network types, represent each label as a ten-value one-hot vector, and reserve 10% of the training data for validation. The test set stays separate until final evaluation.

In [ ]:
x_train, y_train, x_validation, y_validation, x_test, y_test = preprocess_data(
    x_train_raw, y_train_raw, x_test_raw, y_test_raw
)
print('Training:', x_train.shape, y_train.shape)
print('Validation:', x_validation.shape, y_validation.shape)
print('Test:', x_test.shape, y_test.shape)
print('Normalized pixel range:', x_train.min(), 'to', x_train.max())

## 4. Build the neural network

The baseline model flattens each image and passes it through dense layers. Its ten softmax outputs are class probabilities. To try the optional convolutional model, change `model_type` to `cnn`.

In [ ]:
model_type = 'mlp'  # Change to 'cnn' for the optional convolutional model.
model = build_model(model_type=model_type)
model.summary()

## 5. Train the model

Training is limited to 15 epochs. Early stopping watches validation loss and restores the best weights if the model stops improving.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_validation, y_validation),
    epochs=15,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Evaluate on the test set

Accuracy summarizes the fraction of correct predictions. The classification report also shows precision, recall, and F1-score for each digit.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
probabilities = model.predict(x_test, verbose=0)
predictions = np.argmax(probabilities, axis=1)
actual = np.argmax(y_test, axis=1)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2%}')
print(classification_report(actual, predictions, labels=range(10), zero_division=0))

In [ ]:
matrix = confusion_matrix(actual, predictions, labels=range(10))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted digit')
plt.ylabel('True digit')
plt.title('MNIST confusion matrix')
plt.show()

### Inspect a few mistakes

These examples show where the classifier disagrees with the true label. Some handwritten digits are genuinely ambiguous even to people.

In [ ]:
wrong = np.flatnonzero(predictions != actual)[:10]
figure, axes = plt.subplots(2, 5, figsize=(10, 5))
for axis, index in zip(axes.flat, wrong):
    axis.imshow(x_test[index].squeeze(), cmap='gray')
    axis.set_title(f'True: {actual[index]} | Predicted: {predictions[index]}')
    axis.axis('off')
for axis in axes.flat[len(wrong):]:
    axis.axis('off')
plt.tight_layout()
plt.show()

## 7. Save the model and predict a new image

The saved Keras model can be loaded by the command-line prediction script and Streamlit app. The final cell demonstrates prediction from an image path; change the path to your own single-digit image.

In [ ]:
from pathlib import Path

model_path = Path('models/mnist_model.keras')
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save(model_path)
print(f'Saved model to {model_path.resolve()}')

In [ ]:
# Example: replace this path with an image containing one handwritten digit.
image_path = Path('my_digit.png')
if image_path.is_file():
    digit, confidence, class_probabilities = predict_image(image_path, model_path)
    print(f'Predicted digit: {digit}; confidence: {confidence:.2%}')
else:
    print(f'Place a digit image at {image_path.resolve()} to try a custom prediction.')

# The model can also predict an MNIST test example directly:
test_index = 0
test_probabilities = model.predict(x_test[test_index:test_index + 1], verbose=0)[0]
print('Test example true label:', actual[test_index])
print('Test example predicted label:', int(test_probabilities.argmax()))
plt.imshow(x_test[test_index].squeeze(), cmap='gray')
plt.axis('off');
plt.show()